## Materialized and Non-Materialized Views

### Load and Prepare Data

In [0]:
import requests
from pyspark.sql.types import *
from pyspark.sql.functions import *

# Define the current catalog and volume
catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]
volume_base = f"/Volumes/{catalog_name}/default/spark_lab"

# Define schema for sales data
schema = StructType([
    StructField("SalesOrderNumber", StringType()),
    StructField("SalesOrderLineNumber", IntegerType()),
    StructField("OrderDate", DateType()),
    StructField("CustomerName", StringType()),
    StructField("Email", StringType()),
    StructField("Item", StringType()),
    StructField("Quantity", IntegerType()),
    StructField("UnitPrice", FloatType()),
    StructField("Tax", FloatType())
])

# Load data from 2019, 2020, 2021 CSV files
df = spark.read.schema(schema) \
    .option("header", "true") \
    .csv(f'{volume_base}/*_edited.csv')

display(df.limit(10))

### Create the Views Schema to Store Foundation Tables

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS views_lab;

### Store Raw Data as Delta Lake Tables in the Views Schema

In [0]:
# Store the consolidated sales data as a Delta table
raw_data_path = f'/Volumes/{catalog_name}/default/spark_lab/delta/sales_raw'
df.write.format('delta').mode('overwrite').save(raw_data_path)

# Register as SQL table
df.write.format('delta').mode('overwrite').saveAsTable('views_lab.sales_raw')

print("Raw data stored successfully")
print(f"Total records: {df.count()}")

### Create a REGULAR VIEW - Monthly Sales Summary

In [0]:
%sql
CREATE OR REPLACE VIEW views_lab.v_monthly_sales_summary AS
SELECT 
    year(OrderDate) as Year,
    month(OrderDate) as Month,
    COUNT(DISTINCT SalesOrderNumber) as OrderCount,
    SUM(Quantity) as TotalQuantity,
    ROUND(SUM(UnitPrice * Quantity), 2) as TotalSales,
    ROUND(SUM(Tax), 2) as TotalTax
FROM views_lab.sales_raw
GROUP BY year(OrderDate), month(OrderDate)
ORDER BY Year, Month;

### Query the Regular View

In [0]:
%sql
SELECT * FROM views_lab.v_monthly_sales_summary;

### Create a Materialized View - Yearly Sales Summary

In [0]:
%sql
CREATE MATERIALIZED VIEW IF NOT EXISTS views_lab.mv_yearly_sales_summary AS
SELECT 
    year(OrderDate) as Year,
    COUNT(DISTINCT SalesOrderNumber) as OrderCount,
    COUNT(DISTINCT CustomerName) as UniqueCustomers,
    SUM(Quantity) as TotalQuantity,
    ROUND(SUM(UnitPrice * Quantity), 2) as TotalSales,
    ROUND(SUM(Tax), 2) as TotalTax,
    ROUND(AVG(UnitPrice * Quantity), 2) as AvgOrderValue
FROM views_lab.sales_raw
GROUP BY year(OrderDate);

### Query the Materialized View

In [0]:
%sql
SELECT * FROM views_lab.mv_yearly_sales_summary;

### Create Additional Regular View - Customer Revenue Analysis

In [0]:
%sql
CREATE OR REPLACE VIEW views_lab.v_customer_revenue AS
SELECT 
    CustomerName,
    Email,
    COUNT(DISTINCT SalesOrderNumber) as NumberOfOrders,
    ROUND(SUM(UnitPrice * Quantity), 2) as TotalRevenue,
    ROUND(AVG(UnitPrice * Quantity), 2) as AvgOrderValue,
    COUNT(DISTINCT year(OrderDate)) as YearsActive
FROM views_lab.sales_raw
GROUP BY CustomerName, Email
ORDER BY TotalRevenue DESC;

SELECT * FROM views_lab.v_customer_revenue LIMIT 20;

### Create Materialized View - Product Performance Summary

In [0]:
%sql
CREATE MATERIALIZED VIEW IF NOT EXISTS views_lab.mv_product_performance AS
SELECT 
    Item as Product,
    COUNT(*) as SalesCount,
    SUM(Quantity) as UnitsSold,
    ROUND(AVG(UnitPrice), 2) as AvgPrice,
    ROUND(SUM(UnitPrice * Quantity), 2) as TotalRevenue,
    ROUND(SUM(Tax), 2) as TotalTax
FROM views_lab.sales_raw
GROUP BY Item;

SELECT * FROM views_lab.mv_product_performance ORDER BY TotalRevenue DESC;

### Refresh the Materialized View

In [0]:
%sql
REFRESH MATERIALIZED VIEW views_lab.mv_yearly_sales_summary;
REFRESH MATERIALIZED VIEW views_lab.mv_product_performance;

SELECT 'Materialized views refreshed successfully' as Status;

### List All Views and Materialized Views in the Views Schema

In [0]:
%sql
SHOW VIEWS IN views_lab;